# MaleCNS 神经元形态 + 突触可视化

## 怎么用这个 notebook？

**Colab 是什么：** 就是一个在线的 Python 运行环境，不需要在你电脑上安装任何东西。
每个灰色代码框就是一个「代码块」，点击左边的 ▶ 按钮就能运行它，结果会显示在下方。

**运行顺序：从上到下，一个一个运行，不能跳过。**

---
**本 notebook 会做什么：**
- 从 neuPrint 云端直接获取神经元骨架和突触位点（不需要下载任何大文件）
- 用 JRCFIB2022M 标准脑模板显示 Brain + VNC 轮廓
- 生成三个视角的 PNG 图像（正视图 / 侧视图 / 俯视图）


## 第 1 步：安装依赖包

运行一次就好，之后重新打开 notebook 需要再跑一次。大约需要 2-3 分钟，看到 ✅ 表示完成。

In [ ]:
# 安装所有需要的包（在 Colab 里运行大约 2-3 分钟）
!pip install neuprint-python navis flybrains matplotlib -q
print('✅ 安装完成')

## 第 2 步：填写你的 neuPrint Token 和 Body ID

**只需要改这一个代码块里的内容，其他代码块不需要修改。**

- `TOKEN`：你的 neuPrint API Token（去 neuprint.janelia.org 登录后在右上角账户里找）
- `GROUP_A_IDS`：你要分析的神经元 Body ID 列表
- `GROUP_A_LABEL`：给这组神经元起个名字，用于图片文件命名
- `TASK`：选 `'A'` 画单组（骨架+突触前后），选 `'B'` 画两组间定向连接


In [ ]:
# ════════════════════════════════════════════════
#  只需要修改这里，其他代码块不用动
# ════════════════════════════════════════════════

TOKEN = 'YOUR_NEUPRINT_TOKEN_HERE'   # ← 把你的 token 粘贴在这里（替换这段文字）

# 任务选择：'A' = 单组神经元 + 突触前后位点
#           'B' = 两组神经元 + A→B 方向性突触
TASK = 'A'

# Group A 的 Body ID 列表
GROUP_A_IDS    = [520883, 10692, 10165, 556836]
GROUP_A_LABEL  = 'AN05B101'   # 这组神经元的名称（用于文件命名）

# Group B（只在 TASK='B' 时需要填写）
GROUP_B_IDS    = []
GROUP_B_LABEL  = 'GroupB'

# 输出目录（Colab 里文件会保存到这里，之后可以下载）
OUTPUT_DIR = '/content/neuronplot_output'

print(f'任务类型: {TASK}')
print(f'Group A: {GROUP_A_LABEL}，{len(GROUP_A_IDS)} 个神经元: {GROUP_A_IDS}')
if TASK == 'B':
    print(f'Group B: {GROUP_B_LABEL}，{len(GROUP_B_IDS)} 个神经元: {GROUP_B_IDS}')

## 第 3 步：导入库并连接 neuPrint

这一步会连接到 neuPrint 服务器，验证 token 是否有效。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # 非交互模式，适合 Colab 保存图片
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import navis
import navis.interfaces.neuprint as neu
import flybrains

from neuprint import Client, NeuronCriteria, SynapseCriteria, fetch_synapses

# 连接 neuPrint
client = Client(
    'https://neuprint.janelia.org',
    dataset='male-cns:v0.9',
    token=TOKEN
)

# 测试连接
info = client.fetch_version()
print(f'✅ 连接成功！neuPrint 版本: {info}')

# 准备输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ 输出目录: {OUTPUT_DIR}')

## 第 4 步：从 neuPrint 获取神经元骨架

直接从云端拉取 SWC 骨架，不需要下载任何文件到本地。
坐标单位是 **8nm/voxel**，乘以 8 转为 nm 后与 JRCFIB2022M 模板对齐。

In [ ]:
# 收集所有需要获取的 ID
all_ids = list(set(GROUP_A_IDS + (GROUP_B_IDS if TASK == 'B' else [])))
print(f'获取 {len(all_ids)} 个神经元骨架: {all_ids}')

# 从 neuPrint 获取骨架（自动处理格式）
skels_raw = neu.fetch_skeletons(
    NeuronCriteria(bodyId=all_ids),
    with_synapses=False
)
print(f'获取到 {len(skels_raw)} 个骨架')
print(f'坐标单位: {skels_raw[0].units}')

# 转为 nm（× 8），与 JRCFIB2022M 模板对齐
skels_nm = skels_raw * 8
print(f'\n坐标范围（nm，转换后）:')
for s in skels_nm:
    x_range = (s.nodes.x.min(), s.nodes.x.max())
    y_range = (s.nodes.y.min(), s.nodes.y.max())
    print(f'  ID {s.id}: X={x_range[0]:.0f}~{x_range[1]:.0f}  Y={y_range[0]:.0f}~{y_range[1]:.0f} nm')

# 分组
skel_a = skels_nm.idx[GROUP_A_IDS]   # Group A 骨架
skel_b = skels_nm.idx[GROUP_B_IDS] if TASK == 'B' and GROUP_B_IDS else None
print(f'\n✅ Group A: {len(skel_a)} 个骨架')
if skel_b is not None:
    print(f'✅ Group B: {len(skel_b)} 个骨架')

## 第 5 步：从 neuPrint 获取突触位点

- **Task A**：获取 Group A 所有神经元的突触前（Pre）和突触后（Post）位点
- **Task B**：获取 Group A → Group B 方向的突触连接位点

同样是 8nm/voxel，乘以 8 转为 nm。

In [ ]:
print('获取突触数据...')

if TASK == 'A':
    # Task A: 获取 Group A 的所有 pre 和 post 突触位点
    syn_df = fetch_synapses(
        NeuronCriteria(bodyId=GROUP_A_IDS),
        SynapseCriteria(primary_only=True)
    )
    # 坐标 × 8 → nm
    syn_df[['x', 'y', 'z']] = syn_df[['x', 'y', 'z']] * 8

    pre_pts  = syn_df[syn_df['type'] == 'pre'].copy()
    post_pts = syn_df[syn_df['type'] == 'post'].copy()
    print(f'✅ 突触前位点 (Pre):  {len(pre_pts)} 个')
    print(f'✅ 突触后位点 (Post): {len(post_pts)} 个')
    conn_pts = None

else:  # TASK == 'B'
    # Task B: 只获取从 A 到 B 的突触（pre 在 A 中，post 在 B 中）
    syn_all = fetch_synapses(
        NeuronCriteria(bodyId=GROUP_A_IDS),
        SynapseCriteria(primary_only=True)
    )
    syn_all[['x', 'y', 'z']] = syn_all[['x', 'y', 'z']] * 8

    # A→B：A 的 pre 突触，目标 bodyId 在 B 中
    # neuprint 的 pre 记录里有 partner bodyId 字段
    pre_a = syn_all[syn_all['type'] == 'pre'].copy()
    # 筛选目标在 Group B 里的
    if 'partner_bodyId' in pre_a.columns:
        conn_pts = pre_a[pre_a['partner_bodyId'].isin(GROUP_B_IDS)]
    else:
        conn_pts = pre_a   # 如果没有 partner 字段，显示所有 pre 位点
    pre_pts  = None
    post_pts = None
    print(f'✅ A→B 突触位点: {len(conn_pts)} 个')


## 第 6 步：加载 JRCFIB2022M 标准脑模板

这个模板同时包含 Brain 和 VNC 的轮廓，坐标单位是 nm，与骨架对齐。
这一步会从网络下载模板 mesh 文件（第一次需要几秒钟）。

In [ ]:
import flybrains

# 获取 Brain 和 VNC 的 mesh 轮廓
brain_mesh = flybrains.JRCFIB2022M.mesh_brain
vnc_mesh   = flybrains.JRCFIB2022M.mesh_vnc

print('JRCFIB2022M 模板坐标范围（nm）:')
bv = brain_mesh.vertices
vv = vnc_mesh.vertices
print(f'  Brain: X={bv[:,0].min():.0f}~{bv[:,0].max():.0f}  '
      f'Y={bv[:,1].min():.0f}~{bv[:,1].max():.0f}  '
      f'Z={bv[:,2].min():.0f}~{bv[:,2].max():.0f}')
print(f'  VNC:   X={vv[:,0].min():.0f}~{vv[:,0].max():.0f}  '
      f'Y={vv[:,1].min():.0f}~{vv[:,1].max():.0f}  '
      f'Z={vv[:,2].min():.0f}~{vv[:,2].max():.0f}')

# 包装成 navis.Volume 用于叠加绘图
brain_vol = navis.Volume(
    vertices=brain_mesh.vertices, faces=brain_mesh.faces,
    name='Brain', color=(0.6, 0.6, 0.6, 0.12)
)
vnc_vol = navis.Volume(
    vertices=vnc_mesh.vertices, faces=vnc_mesh.faces,
    name='VNC', color=(0.5, 0.7, 0.9, 0.12)
)
print('✅ 模板加载完成')

## 第 7 步：绘图并保存 PNG

会生成三个视角的图像：
- **正视图 (front)**：看脑的正面
- **侧视图 (side)**：看脑的侧面
- **俯视图 (top)**：从上往下看

图像保存在 `/content/neuronplot_output/` 目录下。

In [ ]:
# ── 颜色设置 ─────────────────────────────────────────────────────
COLOR_A    = '#E84848'   # Group A 骨架颜色（红）
COLOR_B    = '#2B5FFF'   # Group B 骨架颜色（蓝）
COLOR_PRE  = '#FF1493'   # 突触前位点颜色（深粉）
COLOR_POST = '#00CED1'   # 突触后位点颜色（青）
COLOR_CONN = '#FF8C00'   # A→B 连接突触颜色（橙）

# ── 视图配置 ─────────────────────────────────────────────────────────
VIEWS = {
    'front': ('x', '-z'),
    'side':  ('y', '-z'),
    'top':   ('x', '-y'),
}

def scatter_syn(ax, pts_df, axis_h, axis_v, color, label, size=3, alpha=0.7):
    """
    在 2D 轴上绘制突触散点。
    注意：不对坐标取负号。
    navis.plot2d 内部通过轴翻转（invert_xaxis/invert_yaxis）来实现 '-z' 这类方向，
    并不是对数据坐标取负。如果这里再乘以 -1，结果就变成了相对 0 点的镜像。
    """
    if pts_df is None or len(pts_df) == 0:
        return
    col_map = {'x':'x', '-x':'x', 'y':'y', '-y':'y', 'z':'z', '-z':'z'}
    col_h = col_map[axis_h]
    col_v = col_map[axis_v]
    ax.scatter(
        pts_df[col_h].values,
        pts_df[col_v].values,
        c=color, s=size, alpha=alpha, linewidths=0,
        label=label, rasterized=True
    )

def plot_one_view(view_name, axis_h, axis_v):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_facecolor('white')
    ax.set_aspect('equal')

    handles = []

    # 1. 模板轮廓（brain 灰 + VNC 浅蓝）
    navis.plot2d([brain_vol, vnc_vol], ax=ax, method='2d',
                 view=(axis_h, axis_v), alpha=0.15, linewidth=0.5)

    # 2. 神经元骨架
    navis.plot2d(skel_a, ax=ax, method='2d',
                 view=(axis_h, axis_v),
                 color=COLOR_A, linewidth=0.8, alpha=0.9)
    handles.append(mpatches.Patch(color=COLOR_A, label=GROUP_A_LABEL))

    if TASK == 'B' and skel_b is not None:
        navis.plot2d(skel_b, ax=ax, method='2d',
                     view=(axis_h, axis_v),
                     color=COLOR_B, linewidth=0.8, alpha=0.9)
        handles.append(mpatches.Patch(color=COLOR_B, label=GROUP_B_LABEL))

    # 3. 突触位点（坐标直接使用，不取负，轴方向由 navis 统一管理）
    if TASK == 'A':
        scatter_syn(ax, pre_pts,  axis_h, axis_v, COLOR_PRE,  'Pre-syn',  size=3)
        scatter_syn(ax, post_pts, axis_h, axis_v, COLOR_POST, 'Post-syn', size=3)
        handles += [
            plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=COLOR_PRE,
                       markersize=8, label='Pre-synaptic'),
            plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=COLOR_POST,
                       markersize=8, label='Post-synaptic'),
        ]
    else:
        scatter_syn(ax, conn_pts, axis_h, axis_v, COLOR_CONN, 'A→B syn', size=4)
        handles.append(
            plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=COLOR_CONN,
                       markersize=8, label='A → B synapses')
        )

    # 4. 图例和标题
    ax.legend(handles=handles, fontsize=8, loc='upper right', framealpha=0.9)
    task_str = f'{GROUP_A_LABEL} + Pre/Post' if TASK=='A' else f'{GROUP_A_LABEL}→{GROUP_B_LABEL}'
    view_labels = {'front':'Front view (XZ)', 'side':'Side view (YZ)', 'top':'Top view (XY)'}
    ax.set_title(f'{task_str}  |  {view_labels[view_name]}', fontsize=12, pad=8)
    ax.set_xlabel(axis_h.lstrip('-') + ' (nm)')
    ax.set_ylabel(axis_v.lstrip('-') + ' (nm)')
    ax.invert_yaxis()

    # 5. 保存
    safe = GROUP_A_LABEL.replace('/', '_').replace(' ', '_')
    fname = f'{OUTPUT_DIR}/{safe}_task{TASK}_{view_name}.png'
    plt.tight_layout()
    fig.savefig(fname, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✅ 保存: {fname}')
    return fname

# ── 生成三个视角 ─────────────────────────────────────────────────────
print('开始绘图...')
saved_files = []
for vname, (ah, av) in VIEWS.items():
    f = plot_one_view(vname, ah, av)
    saved_files.append(f)

print(f'\n全部完成！共生成 {len(saved_files)} 张图')

## 第 8 步：在 notebook 里预览图像

运行下面这个代码块，可以直接在 Colab 里看到生成的图片。
也可以在左侧文件栏（📁 图标）找到 `neuronplot_output` 文件夹，右键下载。

In [ ]:
from IPython.display import display, Image

for f in saved_files:
    print(f'\n--- {f.split("/")[-1]} ---')
    display(Image(filename=f, width=700))

## 下载图像到本地

运行下面的代码块，会把所有图片打成一个 zip 压缩包，然后自动弹出下载。

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/neuronplot_output'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
files.download(zip_path + '.zip')
print('✅ 下载开始（如果没有弹出，请检查浏览器是否阻止了弹窗）')